# RNN 实验：读完整个序列，记住第一个信号

你用 PyTorch 的基础张量运算手写一个最简单的 RNN（不用 `nn.RNN`），让它读完一段数字序列后，判断**第一个数字**是正还是负。数据是自己生成的，所以你可以直接观察网络“记得住多久”。核心函数保留了完整签名、docstring 和 TODO 提示，函数体留给你实现。公式、API 链接、自检单元格和全部绘图代码已经提供。

## 使用方法

1. 在本目录运行 `uv sync`，然后在 VS Code / Jupyter 中选择 `.venv/bin/python` 作为 kernel。
2. 从上往下运行。遇到 `TODO` 函数时先实现它，再运行紧随其后的自检单元格。
3. 修改函数后要重新运行定义它的单元格，编辑器里的代码不会自动同步到 kernel。
4. 没实现的函数会抛出 `NotImplementedError`，这是练习刚开始时的正常状态。
5. 保持函数名、参数名、返回值约定和 `nn.Parameter` 的属性名不变（`W_x`、`W_h`、`b`、`W_y`、`b_y`），自检和绘图依赖它们。
6. 全部完成后执行 Restart Kernel → Run All，确认结果不依赖旧变量。

**规则**：模型的循环和递推公式要自己写。不要在模型里调用 `nn.RNN`、`nn.RNNCell`、`nn.LSTM`。`nn.RNNCell` 只出现在自检里，用来核对你的 `rnn_step`。

## 路线

| Lab | 内容 | 你要实现 | 已提供的观察图 |
| --- | --- | --- | --- |
| 1 | 数据 | `generate_sequences` | 样本序列示例 |
| 2 | RNN 模型 | `SimpleRNN.__init__`, `rnn_step`, `forward` | 三个时间步的计算图 |
| 3 | BPTT 与梯度 | `hidden_grad_norms` | 共享参数的梯度 = 各时间步贡献之和、每个时间步的梯度范数 |
| 4 | 训练流程 | `train_epoch`, `evaluate` | loss / 准确率 / 裁剪前梯度范数曲线 |
| 5 | 长序列实验 | 无（运行并记录） | T=10 / 30 / 100 对比 |

一共 7 个函数 / 方法（`SimpleRNN` 的 `__init__`、`rnn_step`、`forward` 分开计数）。

## 1. Problem Definition

**任务**：每个样本是长度 $T$ 的数字序列。

- 第一个数字 $x_1$ 随机取 $-1$ 或 $+1$，两种情况各占一半。
- 后面 $T-1$ 个数字独立采样自 $\text{Uniform}(-1,1)$，是干扰。
- 网络必须**按顺序读完整个序列**，在最后判断第一个数字是正（标签 1）还是负（标签 0）。

例如输入 `[+1, 0.2, -0.8, 0.4, ...]`，标签是 1。

| 项目 | 本实验的约定 |
| --- | --- |
| 输入 | `[B, T, 1]` |
| 标签 | `[B, 1]`，浮点数 0 或 1 |
| 隐藏状态 | `[B, 32]` |
| 输出 | `[B, 1]`，一个 **logit**（不要在模型里加 Sigmoid） |
| 数据量 | 独立生成 10,000 训练 / 2,000 验证 / 2,000 测试样本 |
| 起始序列长度 | $T=10$ |
| 损失 | `BCEWithLogitsLoss` |
| 训练配置 | Adam，`lr=0.001`，batch 64，先训练 20 轮 |

**为什么这个任务适合研究记忆？** 答案只取决于第一个数字，之后的 $T-1$ 个数字都是噪声。网络必须把第一步的信息一直保留到最后一步，而且不能被中间的噪声冲掉。$T$ 越大，需要保留的时间越长，梯度要穿过的时间步也越多。

**验收**：先跑通 $T=10$，验证准确率超过 95%。然后分别从头训练 $T=30$ 和 $T=100$ 的模型，记录准确率和训练难度。

## 2. 环境与 API 速查

下表只列出实现所需的**部分参数**，不是 API 的全部参数。

### 2.1 创建数据与随机数（Lab 1）

| API（官方文档） | 关键参数 / 返回值 | 用在哪里 |
| --- | --- | --- |
| [`torch.Generator().manual_seed(seed)`](https://docs.pytorch.org/docs/stable/generated/torch.Generator.html) | 独立的随机数生成器，不影响全局随机状态 | 让不同数据集用不同种子、可复现 |
| [`torch.rand(*size, generator=g)`](https://docs.pytorch.org/docs/stable/generated/torch.rand.html) | $[0,1)$ 上的均匀分布；`x * 2 - 1` 变成 $[-1,1)$ | 干扰数字 |
| [`torch.randint(low, high, size, generator=g)`](https://docs.pytorch.org/docs/stable/generated/torch.randint.html) | 整数，**不含 `high`**；`randint(0, 2, ...)` 取 0 或 1 | 随机选符号 |
| [`Tensor.float()`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.float.html)、[`Tensor.unsqueeze(dim)`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.unsqueeze.html) | 转 `float32`；插入一个长度为 1 的维度 | 把标签变成 `[n, 1]` |
| [`TensorDataset`](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.TensorDataset)、[`DataLoader`](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader) | 把张量包装成数据集，按 batch 迭代 | 已提供的 `make_loaders` |

### 2.2 模型（Lab 2、3）

| API（官方文档） | 关键参数 / 返回值 | 用在哪里 |
| --- | --- | --- |
| [`nn.Parameter(tensor)`](https://docs.pytorch.org/docs/stable/generated/torch.nn.parameter.Parameter.html) | 赋给 `self.xxx` 后自动注册为可训练参数 | `W_x`、`W_h`、`b`、`W_y`、`b_y` |
| [`Tensor.uniform_(a, b)`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.uniform_.html) | 原地用 $U(a,b)$ 填充 | 初始化 |
| [`torch.empty(*size)`](https://docs.pytorch.org/docs/stable/generated/torch.empty.html)、[`torch.zeros(*size)`](https://docs.pytorch.org/docs/stable/generated/torch.zeros.html) | 创建张量 | 参数与初始隐藏状态 |
| [`torch.tanh(x)`](https://docs.pytorch.org/docs/stable/generated/torch.tanh.html) | 逐元素双曲正切 | 递推 |
| [`x @ W.T`](https://docs.pytorch.org/docs/stable/generated/torch.matmul.html) | `[B, in] @ [in, out] → [B, out]`；`W.T` 是转置 | 行向量写法的线性层 |
| [`Tensor.new_zeros`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.new_zeros.html)、`torch.zeros(B, H, device=x.device)` | 零隐藏状态，要与输入在同一个 device 上 | 每个 batch 开始时的 $h_0$ |
| [`nn.RNNCell(input_size, hidden_size)`](https://docs.pytorch.org/docs/stable/generated/torch.nn.RNNCell.html) | 属性 `weight_ih`、`weight_hh`、`bias_ih`、`bias_hh`；**有两个 bias** | 只在自检里核对 `rnn_step` |

### 2.3 训练与梯度（Lab 3、4）

| API（官方文档） | 关键参数 / 返回值 | 用在哪里 |
| --- | --- | --- |
| [`nn.BCEWithLogitsLoss()`](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html) | 输入 logits `[B, 1]` 和浮点标签 `[B, 1]`；内部把 Sigmoid 和 BCE 合在一起，数值更稳定 | 损失 |
| [`torch.sigmoid(z)`](https://docs.pytorch.org/docs/stable/generated/torch.sigmoid.html) | logit → 概率；预测 `logits > 0` 等价于 `sigmoid(logits) > 0.5` | 预测 |
| [`nn.utils.clip_grad_norm_(params, max_norm)`](https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.clip_grad_norm_.html) | 原地缩放梯度，使全局范数不超过 `max_norm`；**返回裁剪前的总范数** | 梯度裁剪 |
| [`Tensor.retain_grad()`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.retain_grad.html) | 让非叶子张量（如 $h_t$）在 `backward()` 后保留 `.grad` | 观察每个时间步的梯度 |
| [`torch.autograd.grad(outputs, inputs)`](https://docs.pytorch.org/docs/stable/generated/torch.autograd.grad.html) | 直接返回梯度，不写进 `.grad` | 梯度演示 |
| [`Tensor.norm(dim=1)`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.norm.html) | 沿 `dim` 求 L2 范数 | 每个样本的梯度范数 |
| [`torch.optim.Adam`](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html)、`optimizer.zero_grad()` | 同 CNN 实验 | 优化器 |

官方参考：[`nn.RNNCell`](https://docs.pytorch.org/docs/stable/generated/torch.nn.RNNCell.html)（递推公式）、[`nn.RNN`](https://docs.pytorch.org/docs/stable/generated/torch.nn.RNN.html)（整条序列的版本）。

### 2.4 NumPy 手写、PyTorch、TensorFlow / Keras 对比

本实验用 PyTorch 的自动求导手写 RNN。TensorFlow / Keras 只做对照，不安装、不运行。

| 维度 | NumPy 手写（把 BPTT 也自己写） | PyTorch 手写循环（本实验） | PyTorch `nn.RNN` | TensorFlow / Keras |
| --- | --- | --- | --- | --- |
| 前向 | 循环里用 `np.tanh` | 循环里用 `torch.tanh` | 一次调用处理整条序列 | `SimpleRNN`，或 `RNN(SimpleRNNCell)` |
| 反向传播 | 要手写 4.4 的 BPTT 公式，并保存每步的 $h_t$ | autograd 自动沿时间展开的计算图反传 | 同左，内部用优化的 kernel | `tf.GradientTape` / `fit` |
| 每步中间量 | 自己存 | 在 Python 里随手可取（`retain_grad`） | 只返回输出和最后的 $h$ | `return_sequences=True` 返回所有 $h_t$ |
| 速度 | 最慢 | 慢（每个时间步都是一次 Python 调用） | 快（cuDNN 等融合 kernel，没有 Python 循环） | 快 |
| 调试 | 最透明 | 可以逐步 `print`、断点、取梯度 | 内部不可见 | `@tf.function` 里需要图模式调试技巧 |
| bias | 自定 | 本实验只有一个 `b` | **有两个**：`bias_ih` 和 `bias_hh`（数学上等价于一个） | 只有一个 `bias` |
| 权重布局 | 自定 | `W_x: [H, I]`，计算时用 `x @ W_x.T` | `weight_ih: [H, I]` | `kernel: [I, H]`，计算时用 `x @ kernel`（转置了） |
| 梯度裁剪 | 自己算范数 | `clip_grad_norm_` | 同左 | 优化器的 `global_clipnorm=...` 或 `tf.clip_by_global_norm`（注意 `clipnorm=` 是对每个梯度张量单独裁剪，不是全局范数） |

**取舍**：手写循环适合理解 BPTT 和观察每个时间步，但很慢；真正训练长序列时用 `nn.RNN` / `nn.LSTM` 这样的融合实现。这个实验里 $T=100$ 的手写循环仍然只要几十秒，所以可以直接用。

> **TODO（研究）**：
> 1. 把你的 `SimpleRNN` 换成 `nn.RNN(1, 32, batch_first=True)` + `nn.Linear(32, 1)`，用同样的数据和 T=100，比较每个 epoch 的耗时。
> 2. 读 `nn.RNN` 文档里的 `batch_first` 参数和输出 `output`、`h_n` 的形状，弄清为什么默认的输入是 `[T, B, I]`。
> 3. 用 Keras 的 `SimpleRNN(32)` 写同一个任务，对照它的 `kernel`、`recurrent_kernel`、`bias` 和这里的 $W_x^\top$、$W_h^\top$、$b$。

In [ ]:
import time
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
print("Matplotlib:", matplotlib.__version__)

SERIES_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#4a3aa7"]
HILITE = "#eb6834"

plt.rcParams["font.sans-serif"] = [
    "PingFang SC", "Heiti SC", "Arial Unicode MS", "Microsoft YaHei", "SimHei",
    "Noto Sans CJK SC", "WenQuanYi Micro Hei", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 11,
})

# 序列模型的张量很小，Python 循环占主要时间，用 CPU 反而比 GPU / MPS 更快。
DEVICE = torch.device("cpu")
torch.set_num_threads(4)

## 3. 核心参数配置

| Hyperparameter | 代码常量 | 值 | 说明 |
| --- | --- | --- | --- |
| 序列长度 | `T_START` | 10 | 先在这个长度上跑通 |
| 对比的序列长度 | `T_LIST` | `[10, 30, 100]` | Lab 5 逐个从头训练 |
| 训练 / 验证 / 测试样本数 | `N_TRAIN`, `N_VAL`, `N_TEST` | 10,000 / 2,000 / 2,000 | 三份数据用不同种子独立生成 |
| 输入 / 隐藏 / 输出维度 | `INPUT_SIZE`, `HIDDEN_SIZE`, `OUTPUT_SIZE` | 1 / 32 / 1 | |
| Learning rate | `LEARNING_RATE` | 0.001 | Adam |
| Batch size | `BATCH_SIZE` | 64 | |
| Epochs | `EPOCHS` | 20 | |
| 梯度裁剪阈值 | `CLIP_NORM` | 1.0 | 全局范数超过它就按比例缩小 |
| Random seed | `SEED` | 42 | |

模型共有 $32\times1+32\times32+32+1\times32+1=1121$ 个可训练参数。

In [ ]:
T_START = 10
T_LIST = [10, 30, 100]
N_TRAIN, N_VAL, N_TEST = 10_000, 2_000, 2_000
INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE = 1, 32, 1
LEARNING_RATE = 1e-3
BATCH_SIZE = 64
EPOCHS = 20
CLIP_NORM = 1.0
SEED = 42

n_params = HIDDEN_SIZE * INPUT_SIZE + HIDDEN_SIZE * HIDDEN_SIZE + HIDDEN_SIZE + OUTPUT_SIZE * HIDDEN_SIZE + OUTPUT_SIZE
print("可训练参数数量：", n_params)
assert n_params == 1121

## 4. 公式总表

实现时随时回来查这一节。

### 4.1 符号与形状

代码里**每一行是一个样本**，所以线性变换写成 $xW^\top$，而不是 $Wx$。$B$ 是 batch 大小，$I$ 是输入维度，$H$ 是隐藏维度。

| 数学符号 | 代码变量 | Shape | 含义 |
| --- | --- | --- | --- |
| $x_t$ | `x[:, t]` | $(B, I)$ | 第 $t$ 个时间步的输入 |
| $h_t$ | `h` | $(B, H)$ | 第 $t$ 步之后的隐藏状态；$h_0=\mathbf 0$ |
| $W_x$ | `self.W_x` | $(H, I)$ | 输入到隐藏的权重 |
| $W_h$ | `self.W_h` | $(H, H)$ | 隐藏到隐藏的权重 |
| $b$ | `self.b` | $(H,)$ | 隐藏层偏置 |
| $W_y$ | `self.W_y` | $(O, H)$ | 隐藏到输出的权重 |
| $b_y$ | `self.b_y` | $(O,)$ | 输出偏置 |
| $z$ | `logits` | $(B, O)$ | 输出 logit，$O=1$ |
| $y$ | `y` | $(B, 1)$ | 标签 0 或 1（浮点数） |

### 4.2 RNN 递推（many-to-one）

$$
h_0=\mathbf 0,\qquad
h_t=\tanh\!\left(x_tW_x^\top+h_{t-1}W_h^\top+b\right)\quad(t=1,\dots,T),\qquad
z=h_TW_y^\top+b_y
$$

同一组 $W_x,W_h,b$ 在**所有时间步复用**。只在最后一步 $T$ 用 $h_T$ 做预测，这种结构叫 many-to-one。

**每个 batch 开始时 $h_0$ 必须重新置零**。不同 batch 是互不相关的序列，不能把上一个 batch 的隐藏状态带过来。

### 4.3 损失与预测

$$
\sigma(z)=\frac1{1+e^{-z}},\qquad
\mathcal L=-\frac1B\sum_{n=1}^{B}\Bigl[y^{(n)}\log\sigma(z^{(n)})+\bigl(1-y^{(n)}\bigr)\log\bigl(1-\sigma(z^{(n)})\bigr)\Bigr],\qquad
\hat y=\mathbf 1[z>0]
$$

`BCEWithLogitsLoss` 直接从 logit $z$ 计算，数值稳定。$\partial\mathcal L/\partial z=\bigl(\sigma(z)-y\bigr)/B$。

### 4.4 BPTT（Backpropagation Through Time）

把循环按时间展开，就是一个深度为 $T$ 的前馈网络，只是每一层用同一组权重。记 $a_t=x_tW_x^\top+h_{t-1}W_h^\top+b$（tanh 之前），$\delta_t=\partial\mathcal L/\partial h_t$（形状 $(B,H)$）。反向递推：

$$
\delta_T=\frac{\partial\mathcal L}{\partial z}\,W_y,\qquad
\frac{\partial\mathcal L}{\partial a_t}=\delta_t\odot\bigl(1-h_t^{\,2}\bigr),\qquad
\delta_{t-1}=\frac{\partial\mathcal L}{\partial a_t}\,W_h
$$

共享参数的梯度是**所有时间步贡献之和**：

$$
\frac{\partial\mathcal L}{\partial W_h}=\sum_{t=1}^{T}\Bigl(\frac{\partial\mathcal L}{\partial a_t}\Bigr)^{\!\top}h_{t-1},\qquad
\frac{\partial\mathcal L}{\partial W_x}=\sum_{t=1}^{T}\Bigl(\frac{\partial\mathcal L}{\partial a_t}\Bigr)^{\!\top}x_t,\qquad
\frac{\partial\mathcal L}{\partial b}=\sum_{t=1}^{T}\sum_{n=1}^{B}\frac{\partial\mathcal L}{\partial a_t}
$$

形状检查：$(H,B)\cdot(B,H)=(H,H)$，$(H,B)\cdot(B,I)=(H,I)$。

**整条序列只需要对最终 loss 调用一次 `backward()`**。autograd 会沿着展开的计算图一路回传，并把同一个参数在各时间步的梯度自动累加。

### 4.5 梯度消失与梯度爆炸

把 4.4 的递推连乘起来，第 $k$ 步的梯度和最后一步的梯度之间隔着 $T-k$ 个因子：

$$
\delta_k=\delta_T\prod_{t=k+1}^{T}\Bigl(\mathrm{diag}\bigl(1-h_t^{\,2}\bigr)\,W_h\Bigr)\quad(\text{行向量右乘，顺序为 }t=T,T-1,\dots,k+1)
$$

每个因子的大小大致是 $\gamma\,\|W_h\|$，其中 $\gamma=\max|1-h_t^2|\le1$：

$$
\|\delta_k\|\;\lesssim\;\|\delta_T\|\,\bigl(\gamma\,\|W_h\|\bigr)^{T-k}
$$

- $\gamma\|W_h\|<1$：梯度**指数衰减**（消失）。网络无法把 $T-k$ 步之后的 loss 归因到第 $k$ 步，所以学不会远距离依赖。
- $\gamma\|W_h\|>1$：上界不再限制梯度，梯度**可能指数增长**（爆炸），更新步长失控，loss 出现 NaN 或剧烈抖动。注意上面只是上界：它能推出“小于 1 时一定衰减”，推不出“大于 1 时一定爆炸”。
- tanh 的导数 $1-h^2\le1$，只会让缩小的方向更严重。

这个实验里第一个数字距离最后一步 $T-1$ 步，所以 $T$ 越大梯度消失越明显。

### 4.6 梯度裁剪（针对爆炸）

对所有参数的梯度拼起来求全局 L2 范数 $\|g\|$，超过阈值 $c$ 就整体缩小，**方向不变**：

$$
g\leftarrow g\cdot\min\!\left(1,\frac{c}{\|g\|+10^{-6}}\right)
$$

裁剪只能缓解**爆炸**，对**消失**没有帮助（缩小一个本来就很小的梯度没有意义）。`clip_grad_norm_` 会返回**裁剪前**的 $\|g\|$，记录它可以观察训练过程中梯度是否爆炸。

In [ ]:
# 参考图：三个时间步的计算图。你应该先自己在纸上画一遍，再对照这张图。
def draw_unrolled_graph(T=3):
    fig, ax = plt.subplots(figsize=(11, 5.2))
    ax.set_xlim(0, 12); ax.set_ylim(-2.0, 6.2); ax.axis("off"); ax.grid(False)

    def box(x, y, text, color, w=1.25, h=0.75, fs=11):
        ax.add_patch(FancyBboxPatch((x - w / 2, y - h / 2), w, h, boxstyle="round,pad=0.04", fc="white", ec=color, lw=2))
        ax.text(x, y, text, ha="center", va="center", fontsize=fs)

    def arrow(p, q, color="#555", ls="-", rad=0.0, lw=1.6):
        ax.add_patch(FancyArrowPatch(p, q, arrowstyle="-|>", mutation_scale=14, color=color, ls=ls, lw=lw,
                                     connectionstyle=f"arc3,rad={rad}", shrinkA=2, shrinkB=2))

    xs = [2.6 + 2.6 * k for k in range(T)]               # 时间步的 x 位置
    box(0.9, 2.6, r"$h_0=\mathbf{0}$", "#888")
    for k, x in enumerate(xs, start=1):
        box(x, 2.6, rf"$h_{k}$" + "\n" + r"tanh", SERIES_COLORS[0], h=1.0)
        box(x, 0.2, rf"$x_{k}$", "#888")
        arrow((x, 0.6), (x, 2.05), color="#888")
        prev = 0.9 if k == 1 else xs[k - 2]
        arrow((prev + 0.65, 2.6), (x - 0.65, 2.6), color=SERIES_COLORS[0])
    # 共享参数：三个颜色对应三类参数，每个时间步都有一条虚线指进来
    for name, color, y, label in [("W_x", SERIES_COLORS[1], 5.6, r"$W_x$（输入权重）"),
                                  ("W_h", SERIES_COLORS[2], 4.8, r"$W_h$（循环权重）"),
                                  ("b", SERIES_COLORS[3], 4.0, r"$b$（偏置）")]:
        box(0.9, y, label, color, w=1.6, h=0.6, fs=9)
        for x in xs:
            arrow((1.7, y), (x - 0.1, 3.15), color=color, ls="--", rad=-0.05, lw=1.1)
    # 输出
    xo = xs[-1] + 2.3
    box(xo, 2.6, r"$z=h_TW_y^\top+b_y$", SERIES_COLORS[4], w=2.0, fs=10)
    arrow((xs[-1] + 0.65, 2.6), (xo - 1.0, 2.6), color=SERIES_COLORS[0])
    box(xo, 0.6, r"$\mathcal{L}$" + "\nBCEWithLogits", HILITE, w=1.8, h=0.9, fs=9)
    arrow((xo, 2.2), (xo, 1.1), color="#555")
    # 反向传播的梯度（红色）
    for k in range(T, 1, -1):
        arrow((xs[k - 1] - 0.65, 3.05), (xs[k - 2] + 0.65, 3.05), color=HILITE, rad=-0.35, lw=1.4)
    ax.text(xs[1], -1.3, "橙色弯箭头：梯度沿时间向前传（BPTT）。每个虚线箭头指向同一个参数，\n所以 $W_x$、$W_h$、$b$ 的梯度是 T 个时间步贡献之和", ha="center", fontsize=9.5, color="#444")
    ax.set_title(f"T={T} 的展开计算图：同一组 $W_x, W_h, b$ 被每个时间步重复使用", fontsize=12)
    plt.show()


draw_unrolled_graph(3)

> **你来做（手画）**：合上上面的图，在纸上画出 $T=3$ 的计算图：节点有 $x_1,x_2,x_3$、$h_0,\dots,h_3$、$z$、$\mathcal L$，以及 $W_x,W_h,b$。然后用红笔画出反向传播时梯度的路径，标出 $W_h$ 的梯度从哪些边汇入。
>
> **回答（写在这里）**：为什么共享参数会收到多个时间步的梯度贡献？（提示：在计算图里 $W_h$ 对应几条边？链式法则对“同一个变量出现在多处”怎么处理？）
>
> 你的回答：

## Lab 1：生成序列数据

**你要实现**：`generate_sequences(n, T, seed)`

- 返回 `X`，形状 `[n, T, 1]`，`float32`：
  - `X[:, 0, 0]` 随机取 $-1$ 或 $+1$，各一半的概率；
  - `X[:, 1:, 0]` 独立采样自 $\text{Uniform}(-1,1)$。
- 返回 `y`，形状 `[n, 1]`，`float32`，等于 `1.0` 当且仅当第一个数字是 $+1$。
- 用 `torch.Generator().manual_seed(seed)` 控制随机性，同一个 `seed` 必须得到完全相同的数据，不要改动全局随机状态。

**提示**：

- `torch.rand(n, T, 1, generator=g) * 2 - 1` 得到 $[-1,1)$ 上的均匀分布。
- `torch.randint(0, 2, (n,), generator=g)` 得到 0 或 1，`* 2 - 1` 变成 $-1$ 或 $+1$。
- 标签是 `first > 0`，转成 `float` 再变成 `[n, 1]`。
- 用同一个 generator **连续**调用：先生成干扰，再生成符号，顺序固定，结果才可复现。

In [ ]:
def generate_sequences(n: int, T: int, seed: int) -> tuple[torch.Tensor, torch.Tensor]:
    """生成 n 条长度为 T 的序列和标签。

    Parameters
    ----------
    n : int
        样本数。
    T : int
        序列长度，必须 >= 2。
    seed : int
        随机种子。

    Returns
    -------
    X : Tensor, shape (n, T, 1), float32
        第一步是 -1 或 +1，其余是 Uniform(-1, 1) 的干扰。
    y : Tensor, shape (n, 1), float32
        第一个数字为 +1 时是 1.0，否则是 0.0。
    """
    # TODO: g = torch.Generator().manual_seed(seed)
    # TODO: X = torch.rand(n, T, 1, generator=g) * 2 - 1
    # TODO: 随机符号 first（-1 或 +1），写入 X[:, 0, 0]
    # TODO: y = (first > 0).float().unsqueeze(1)
    raise NotImplementedError("TODO Lab 1: implement generate_sequences")

In [ ]:
# Lab 1 自检
X, y = generate_sequences(10_000, 10, seed=0)
assert X.shape == (10_000, 10, 1) and X.dtype == torch.float32, f"X: {tuple(X.shape)} {X.dtype}"
assert y.shape == (10_000, 1) and y.dtype == torch.float32, f"y: {tuple(y.shape)} {y.dtype}"
assert set(X[:, 0, 0].unique().tolist()) == {-1.0, 1.0}, "第一个数字只能是 -1 或 +1"
assert 0.47 < (X[:, 0, 0] > 0).float().mean() < 0.53, "正负应该各占一半"
assert (X[:, 1:, 0] >= -1).all() and (X[:, 1:, 0] < 1).all(), "干扰必须在 [-1, 1) 内"
assert len(X[:, 1:, 0].unique()) > 1000, "干扰应该是连续随机数，而不是常数"
assert torch.equal(y, (X[:, :1, 0] > 0).float()), "标签应该由第一个数字决定"
# 干扰要和标签无关，否则网络不用记第一个数字就能猜
assert abs(X[y[:, 0] == 1, 1:, 0].mean()) < 0.02 and abs(X[y[:, 0] == 0, 1:, 0].mean()) < 0.02
np.testing.assert_allclose(X[:, 1:, 0].std().item(), (2 / np.sqrt(12)), atol=0.01)   # Uniform(-1, 1) 的标准差是 1/sqrt(3)

X2, y2 = generate_sequences(10_000, 10, seed=0)
assert torch.equal(X, X2) and torch.equal(y, y2), "同一个 seed 应该得到同一份数据"
X3, _ = generate_sequences(10_000, 10, seed=1)
assert not torch.equal(X, X3), "不同 seed 应该得到不同的数据"
state_before = torch.get_rng_state()
generate_sequences(10, 5, seed=0)
assert torch.equal(state_before, torch.get_rng_state()), "不要改动全局随机状态，请使用 torch.Generator"
# 长度不同时，序列依然合法
for T_ in (2, 30, 100):
    Xt, yt = generate_sequences(100, T_, seed=3)
    assert Xt.shape == (100, T_, 1) and yt.shape == (100, 1)
print("Lab 1 generate_sequences 自检通过。")

In [ ]:
def make_loaders(T: int, seed: int = SEED, batch_size: int = BATCH_SIZE):
    """用三个不同的种子独立生成训练 / 验证 / 测试集，并包装成 DataLoader（已提供）。"""
    data = {}
    for offset, (name, n) in enumerate([("train", N_TRAIN), ("val", N_VAL), ("test", N_TEST)]):
        data[name] = generate_sequences(n, T, seed=seed * 1000 + offset)
    train_loader = DataLoader(TensorDataset(*data["train"]), batch_size=batch_size, shuffle=True,
                              generator=torch.Generator().manual_seed(seed))
    val_loader = DataLoader(TensorDataset(*data["val"]), batch_size=batch_size)
    test_loader = DataLoader(TensorDataset(*data["test"]), batch_size=batch_size)
    return train_loader, val_loader, test_loader, data


train_loader, val_loader, test_loader, data10 = make_loaders(T_START)
xb, yb = next(iter(train_loader))
print("一个 batch：", tuple(xb.shape), tuple(yb.shape))
assert xb.shape == (BATCH_SIZE, T_START, 1) and yb.shape == (BATCH_SIZE, 1)

# 观察图：几条样本序列。橙色圆点是第一个数字（答案藏在这里），灰色是干扰。
fig, axes = plt.subplots(2, 3, figsize=(12, 5.0), sharey=True)
Xs, ys = data10["train"]
for ax, idx in zip(axes.ravel(), range(6)):
    seq = Xs[idx, :, 0].numpy()
    ax.plot(range(1, T_START + 1), seq, "-o", color="#999", ms=4, lw=1)
    ax.plot([1], [seq[0]], "o", color=HILITE, ms=10, zorder=5)
    ax.axhline(0, color="#bbb", lw=0.8)
    ax.set_title(f"标签 {int(ys[idx].item())}：第一个数字 {seq[0]:+.0f}")
    ax.set_xlabel("时间步 t"); ax.set_xticks(range(1, T_START + 1))
axes[0, 0].set_ylabel("x_t"); axes[1, 0].set_ylabel("x_t")
fig.suptitle("样本序列：网络要读到最后一步，才能回答第一步是正还是负", y=1.0)
fig.tight_layout(); plt.show()

## Lab 2：实现 RNN 的一步和整个序列

**你要实现** `SimpleRNN` 的三个部分：

| 部分 | 要求 |
| --- | --- |
| `__init__(input_size, hidden_size, output_size)` | 创建 5 个 `nn.Parameter`：`W_x (H, I)`、`W_h (H, H)`、`b (H,)`、`W_y (O, H)`、`b_y (O,)`。权重用 $U(-1/\sqrt H,\,1/\sqrt H)$ 均匀初始化，两个 bias 初始化为 0。这和 `nn.RNN` 的默认初始化范围一致。 |
| `rnn_step(x_t, h_prev)` | 4.2 的一步递推：`x_t` 是 `(B, I)`，`h_prev` 是 `(B, H)`，返回 `(B, H)` 的 $h_t$。 |
| `forward(x, return_states=False)` | `x` 是 `(B, T, I)`。**在函数内部**把 $h_0$ 置零，循环 $T$ 步，用最后一步的 $h_T$ 算输出。`return_states=False` 时返回 logits `(B, O)`；`True` 时返回 `(logits, states)`，`states` 是长度为 $T$ 的 **list**，第 $t$ 个元素是 $h_{t+1}$（形状 `(B, H)`）。 |

**为什么 `states` 是 list 而不是 `torch.stack` 之后的一个张量？** Lab 3 要对每个 $h_t$ 调用 `retain_grad()`，观察 `loss.backward()` 之后每个 $h_t$ 的梯度。`torch.stack` 出来的新张量不在 loss 的计算图上，它的梯度是空的。必须保留计算图里**原来的** $h_t$ 对象。

**提示**：

- 零初始隐藏状态：`h = x.new_zeros(x.size(0), self.hidden_size)`，这样它自动和 `x` 在同一个 device、同一个 dtype 上。
- 循环里用 `x[:, t]` 取第 $t$ 步，形状是 `(B, I)`。
- 不要在循环里调用 `.detach()`，否则梯度传不回更早的时间步。
- 不要在 `__init__` 或模块属性里保存隐藏状态，每次 `forward` 都从零开始。

In [ ]:
class SimpleRNN(nn.Module):
    """手写的 many-to-one RNN 分类器。

    Input : x [B, T, input_size]
    Output: logits [B, output_size]
    """

    def __init__(self, input_size: int = INPUT_SIZE, hidden_size: int = HIDDEN_SIZE, output_size: int = OUTPUT_SIZE) -> None:
        # TODO: super().__init__()；保存 self.hidden_size；k = 1 / sqrt(hidden_size)；
        #       用 nn.Parameter(torch.empty(...).uniform_(-k, k)) 创建 W_x, W_h, W_y；
        #       用 nn.Parameter(torch.zeros(...)) 创建 b, b_y。
        raise NotImplementedError("TODO Lab 2: implement SimpleRNN.__init__")

    def rnn_step(self, x_t: torch.Tensor, h_prev: torch.Tensor) -> torch.Tensor:
        """h_t = tanh(x_t W_x^T + h_{t-1} W_h^T + b)。x_t: (B, I)，h_prev: (B, H)，返回 (B, H)。"""
        # TODO: 4.2 的递推，一行即可。
        raise NotImplementedError("TODO Lab 2: implement SimpleRNN.rnn_step")

    def forward(self, x: torch.Tensor, return_states: bool = False):
        """读完整个序列，用最后的隐藏状态输出 logits。

        return_states=True 时返回 (logits, states)，states 是长度 T 的 list，元素形状 (B, H)。
        """
        # TODO: h = 零状态；states = []；for t in range(T): h = self.rnn_step(x[:, t], h)；记录每一步的 h。
        # TODO: logits = h @ self.W_y.T + self.b_y。
        raise NotImplementedError("TODO Lab 2: implement SimpleRNN.forward")

In [ ]:
# Lab 2 自检 A：参数的形状、初始化
torch.manual_seed(SEED)
rnn = SimpleRNN()
shapes = {name: tuple(p.shape) for name, p in rnn.named_parameters()}
assert shapes == {"W_x": (32, 1), "W_h": (32, 32), "b": (32,), "W_y": (1, 32), "b_y": (1,)}, f"参数形状不对：{shapes}"
assert sum(p.numel() for p in rnn.parameters()) == 1121
k = 1 / np.sqrt(32)
for name in ("W_x", "W_h", "W_y"):
    w = getattr(rnn, name)
    assert w.abs().max() <= k + 1e-6 and w.abs().max() > 0.5 * k, f"{name} 应该来自 U(-1/sqrt(H), 1/sqrt(H))"
assert (rnn.b == 0).all() and (rnn.b_y == 0).all(), "bias 应该初始化为 0"
assert all(p.requires_grad for p in rnn.parameters())

# Lab 2 自检 B：rnn_step —— 先用可以心算的特例
tiny = SimpleRNN(input_size=1, hidden_size=2, output_size=1)
with torch.no_grad():
    tiny.W_x.copy_(torch.tensor([[1.0], [-1.0]])); tiny.W_h.zero_(); tiny.b.copy_(torch.tensor([0.0, 0.5]))
h1 = tiny.rnn_step(torch.tensor([[1.0]]), torch.zeros(1, 2))
np.testing.assert_allclose(h1.detach().numpy(), np.tanh([[1.0, -0.5]]), atol=1e-6)    # tanh([1*1+0, -1*1+0.5])
with torch.no_grad():
    tiny.W_h.copy_(torch.tensor([[0.0, 1.0], [2.0, 0.0]]))                              # 非对称矩阵：能发现漏掉 .T 的错误
h2 = tiny.rnn_step(torch.tensor([[0.0]]), torch.tensor([[0.3, -0.2]]))
np.testing.assert_allclose(h2.detach().numpy(), np.tanh([[-0.2, 0.6 + 0.5]]), atol=1e-6)  # h @ W_h.T = [[-0.2, 0.6]]

# Lab 2 自检 C：rnn_step 与 nn.RNNCell 核对（把同一组权重复制过去）
cell = nn.RNNCell(INPUT_SIZE, HIDDEN_SIZE)       # 默认 tanh；有两个 bias
with torch.no_grad():
    cell.weight_ih.copy_(rnn.W_x); cell.weight_hh.copy_(rnn.W_h)
    cell.bias_ih.copy_(rnn.b); cell.bias_hh.zero_()
xt, hp = torch.randn(7, 1), torch.randn(7, 32)
torch.testing.assert_close(rnn.rnn_step(xt, hp), cell(xt, hp), atol=1e-6, rtol=1e-5)
assert rnn.rnn_step(xt, hp).shape == (7, 32)
print("Lab 2 自检 A–C 通过：参数、rnn_step 与 nn.RNNCell 一致。")

In [ ]:
# Lab 2 自检 D：forward —— 形状、与 nn.RNNCell 循环逐步核对、每个 batch 独立
# 本单元格重新创建 rnn 和 cell，不依赖上一个单元格
torch.manual_seed(SEED)
rnn = SimpleRNN()
cell = nn.RNNCell(INPUT_SIZE, HIDDEN_SIZE)
with torch.no_grad():
    cell.weight_ih.copy_(rnn.W_x); cell.weight_hh.copy_(rnn.W_h)
    cell.bias_ih.copy_(rnn.b); cell.bias_hh.zero_()
xb = torch.randn(5, 12, 1)
logits, states = rnn(xb, return_states=True)
assert logits.shape == (5, 1), f"logits 应为 (5, 1)，得到 {tuple(logits.shape)}"
assert isinstance(states, list) and len(states) == 12 and all(s.shape == (5, 32) for s in states)
torch.testing.assert_close(rnn(xb), logits)

h = torch.zeros(5, 32)
for t in range(12):
    h = cell(xb[:, t], h)
    torch.testing.assert_close(states[t], h, atol=1e-5, rtol=1e-4, msg=lambda m: f"第 {t + 1} 步的 h 不一致\n{m}")
torch.testing.assert_close(logits, h @ rnn.W_y.T + rnn.b_y, atol=1e-5, rtol=1e-4)

# 每个 batch 的隐藏状态都从零开始：同一条序列单独跑，和放在 batch 里跑，结果相同；重复调用结果相同
torch.testing.assert_close(rnn(xb[2:3]), rnn(xb)[2:3], atol=1e-6, rtol=1e-5)
torch.testing.assert_close(rnn(xb), rnn(xb))
assert not hasattr(rnn, "h") and not hasattr(rnn, "hidden"), "不要把隐藏状态保存在模型对象里"

# BPTT 连通性：loss 对第一个时间步的输入有梯度，说明计算图一直连到了 t=1（没有 detach）
xg = xb.clone().requires_grad_(True)
rnn(xg).sum().backward()
assert xg.grad[:, 0].abs().sum() > 0, "第一个时间步收不到梯度：循环里是不是 detach 了 h？"
assert all(torch.isfinite(p.grad).all() for p in rnn.parameters())
print("Lab 2 自检 D 通过：forward 形状正确，与 nn.RNNCell 逐步一致，梯度能回传到第一个时间步。")

## Lab 3：BPTT 与梯度观察

### 3.A 共享参数的梯度 = 各时间步贡献之和（演示，已提供）

下面的单元格用一个 $T=3$ 的小网络验证 4.4 的结论：

1. 正常前向，$W_h$ 在三个时间步复用，得到 `W_h.grad`。
2. 另一种写法：把 $W_h$ 拷贝成 **3 份互相独立的变量** $W_h^{(1)},W_h^{(2)},W_h^{(3)}$，每个时间步用自己的那份。分别求 3 份的梯度。

两者的结论：**共享版的梯度 = 三份独立版的梯度之和**。这就是共享参数会收到多个时间步贡献的原因。这个单元格只用纯张量运算，不依赖你写的模型。

注意输出里**时间步 1 的贡献恰好是 0**：第 1 步用到的是 $h_0=\mathbf 0$，而 $\partial\mathcal L/\partial W_h$ 里每一项都含有 $h_{t-1}$（见 4.4），所以 $t=1$ 这一项为 0。

In [ ]:
torch.manual_seed(0)
H, T3 = 4, 3
x3 = torch.randn(2, T3, 1)
y3 = torch.tensor([[1.0], [0.0]])
Wx = torch.randn(H, 1) * 0.5; Wh = torch.randn(H, H) * 0.5; bb = torch.zeros(H); Wy = torch.randn(1, H) * 0.5

def unrolled_loss(Wh_per_step):
    """Wh_per_step 是 3 个 Wh（共享版是同一个张量重复 3 次，独立版是 3 个不同的张量）。"""
    h = torch.zeros(2, H)
    for t in range(T3):
        h = torch.tanh(x3[:, t] @ Wx.T + h @ Wh_per_step[t].T + bb)
    return nn.functional.binary_cross_entropy_with_logits(h @ Wy.T, y3)

Wh_shared = Wh.clone().requires_grad_(True)
unrolled_loss([Wh_shared] * T3).backward()

Wh_copies = [Wh.clone().requires_grad_(True) for _ in range(T3)]
unrolled_loss(Wh_copies).backward()

per_step = [c.grad for c in Wh_copies]
for t, g in enumerate(per_step, start=1):
    print(f"时间步 {t} 对 W_h 梯度的贡献：Frobenius 范数 {g.norm():.4f}")
print(f"共享版 W_h.grad 的范数：{Wh_shared.grad.norm():.4f}")
torch.testing.assert_close(Wh_shared.grad, sum(per_step))
print("验证通过：共享参数的梯度 = 各时间步独立贡献之和。")

### 3.B 每个时间步的梯度范数（你来实现）

要观察梯度消失，需要看 loss 对**每一个隐藏状态** $h_t$ 的梯度 $\delta_t=\partial\mathcal L/\partial h_t$ 有多大。

**你要实现**：`hidden_grad_norms(model, x, y, loss_fn)`，返回形状 `(T,)` 的张量，第 $t$ 个值是 $\|\delta_t\|$ 在 batch 内的**平均 L2 范数**（每个样本一个范数，再对样本取平均）。

步骤：

1. `logits, states = model(x, return_states=True)`。
2. 对 `states` 里每个张量调用 `retain_grad()`。
3. 算 loss，`model.zero_grad()`，`loss.backward()`。
4. 每个 `h.grad` 的形状是 `(B, H)`。因为 `loss_fn` 取了 batch 平均，梯度里带有 $1/B$，请乘回 `B`，得到“单个样本的 loss 对 $h_t$ 的梯度”。
5. 用 `h.grad.norm(dim=1)` 求每个样本的范数，再 `.mean()`。

这个函数以后在 LSTM 实验里还会复用。

In [ ]:
def hidden_grad_norms(
    model: nn.Module,
    x: torch.Tensor,
    y: torch.Tensor,
    loss_fn: nn.Module,
) -> torch.Tensor:
    """每个时间步隐藏状态的梯度范数 ||dL/dh_t||（单样本 loss，batch 内平均）。

    Returns
    -------
    Tensor, shape (T,)
        第 t 个元素对应 h_t（t 从 1 开始），已 detach。
    """
    # TODO: 见上面的 5 个步骤。注意 retain_grad 要在 backward 之前调用。
    raise NotImplementedError("TODO Lab 3: implement hidden_grad_norms")

In [ ]:
# Lab 3 自检：用 4.4 的递推公式从 h_t 手动算 δ_{t-1}，核对 autograd 给出的梯度
torch.manual_seed(1)
m = SimpleRNN()
xs_, ys_ = generate_sequences(16, 6, seed=5)
bce = nn.BCEWithLogitsLoss()

norms = hidden_grad_norms(m, xs_, ys_, bce)
assert norms.shape == (6,), f"应返回 (T,) = (6,)，得到 {tuple(norms.shape)}"
assert not norms.requires_grad and torch.isfinite(norms).all() and (norms > 0).all()

# 手算：δ_T = (σ(z) - y) W_y（单样本）；δ_{t-1} = (δ_t ⊙ (1 - h_t²)) W_h
with torch.no_grad():
    logits_, states_ = m(xs_, return_states=True)
    delta = (torch.sigmoid(logits_) - ys_) @ m.W_y                    # (B, H)，已是单样本的梯度
    expected = [delta.norm(dim=1).mean()]
    for t in range(5, 0, -1):                                          # t = T, ..., 2
        delta = (delta * (1 - states_[t] ** 2)) @ m.W_h
        expected.append(delta.norm(dim=1).mean())
expected = torch.stack(expected[::-1])
torch.testing.assert_close(norms, expected, atol=1e-5, rtol=1e-4)
print("Lab 3 hidden_grad_norms 自检通过：与手算的 BPTT 递推一致。")
print("各时间步梯度范数（未训练的模型，T=6）：", [round(v, 4) for v in norms.tolist()])

### 3.C 观察：梯度怎样随时间衰减（已提供）

在 $T=30$ 的序列上，看未训练的模型里 $\|\delta_t\|$ 怎样随“离最后一步越远”变化。**纵轴是对数坐标**。如果曲线向左斜着下降，说明越早的时间步收到的梯度越小，这就是梯度消失。下面用三个随机初始化各画一条，结论只针对这类初始化尺度（$U(-1/\sqrt H,1/\sqrt H)$）。

In [ ]:
Xg, yg = generate_sequences(256, 30, seed=7)
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for seed_, ls, color in [(0, "-", SERIES_COLORS[0]), (1, "--", SERIES_COLORS[1]), (2, ":", SERIES_COLORS[2])]:
    torch.manual_seed(seed_)
    g_norms = hidden_grad_norms(SimpleRNN(), Xg, yg, nn.BCEWithLogitsLoss())
    ax.plot(range(1, 31), g_norms.numpy(), ls, color=color, lw=2, label=f"随机初始化 {seed_}")
    print(f"初始化 {seed_}：第 1 步与第 30 步的梯度之比 {(g_norms[0] / g_norms[-1]).item():.2e}")
ax.set_yscale("log"); ax.set_xlabel("时间步 t（30 是最后一步，1 是第一个数字所在的步）")
ax.set_ylabel(r"$\|\partial L/\partial h_t\|$（对数坐标）")
ax.set_title("未训练的 RNN（这三个初始化）：越早的时间步，梯度越小")
ax.legend(); fig.tight_layout(); plt.show()

## Lab 4：训练流程（梯度裁剪与范数记录）

**你要实现**：

| 函数 | 作用 |
| --- | --- |
| `train_epoch(model, loader, loss_fn, optimizer, clip_norm)` | 训练一个 epoch，返回 `(avg_loss, accuracy, grad_norms)`，`grad_norms` 是每个 batch **裁剪前**的梯度总范数组成的 list |
| `evaluate(model, loader, loss_fn)` | 评估，返回 `(avg_loss, accuracy)` |

训练一个 batch 的顺序：

```
logits = model(x)                     # 每个 batch 的 h_0 在 forward 里置零
loss = loss_fn(logits, y)
optimizer.zero_grad()
loss.backward()                       # 整条序列接成一张图，只调用一次
norm = clip_grad_norm_(model.parameters(), clip_norm)   # 返回裁剪前的范数
optimizer.step()
```

**提示**：

- `clip_grad_norm_` 必须在 `backward()` 之后、`step()` 之前调用。它返回一个 Tensor，用 `.item()` 转成 float 再记录。
- 预测：`(logits > 0).float()`，和 `y` 比较得到正确个数。
- 按样本数加权累计 loss（`loss.item() * x.size(0)`）。
- `evaluate` 要 `model.eval()` 和 `torch.no_grad()`。这个模型没有 Dropout，不过养成习惯。

In [ ]:
def train_epoch(
    model: nn.Module,
    loader: DataLoader,
    loss_fn: nn.Module,
    optimizer: torch.optim.Optimizer,
    clip_norm: float,
) -> tuple[float, float, list[float]]:
    """训练一个 epoch。

    Returns
    -------
    avg_loss : float
    accuracy : float
    grad_norms : list[float]
        每个 batch 裁剪前的梯度总范数（clip_grad_norm_ 的返回值）。
    """
    # TODO: model.train()；遍历 loader；前向 → loss → zero_grad → backward → clip_grad_norm_ → step。
    raise NotImplementedError("TODO Lab 4: implement train_epoch")

In [ ]:
def evaluate(model: nn.Module, loader: DataLoader, loss_fn: nn.Module) -> tuple[float, float]:
    """在 loader 上评估，返回 (avg_loss, accuracy)。"""
    # TODO: model.eval()；no_grad；累计 loss 与正确个数。
    raise NotImplementedError("TODO Lab 4: implement evaluate")

In [ ]:
# Lab 4 自检
import copy
torch.manual_seed(SEED)
m = SimpleRNN()
Xs_, ys_ = generate_sequences(256, 10, seed=11)
one_batch = DataLoader(TensorDataset(Xs_[:64], ys_[:64]), batch_size=64)
bce = nn.BCEWithLogitsLoss()

# (1) 返回的 grad_norms 是“裁剪前”的范数：用一个极小的阈值，范数仍应等于手动算出的未裁剪范数
probe = copy.deepcopy(m)
bce(probe(Xs_[:64]), ys_[:64]).backward()
manual_norm = torch.sqrt(sum((p.grad ** 2).sum() for p in probe.parameters())).item()
opt = torch.optim.SGD(m.parameters(), lr=0.0)       # lr=0：只看范数，不改参数
_, _, norms = train_epoch(m, one_batch, bce, opt, clip_norm=1e-6)
assert len(norms) == 1 and isinstance(norms[0], float), "grad_norms 应该是每个 batch 一个 float"
np.testing.assert_allclose(norms[0], manual_norm, rtol=1e-4, err_msg="记录的应该是裁剪前的范数（clip_grad_norm_ 的返回值）")

# (2) 裁剪生效：step 之前梯度范数被缩到 clip_norm 以内（用 hook 在 step 之前读取）
seen = []
class SpyOptimizer(torch.optim.SGD):
    def step(self, *a, **k):
        seen.append(torch.sqrt(sum((p.grad ** 2).sum() for g in self.param_groups for p in g["params"])).item())
        return super().step(*a, **k)
m2 = copy.deepcopy(m)
spy = SpyOptimizer(m2.parameters(), lr=0.0)
train_epoch(m2, one_batch, bce, spy, clip_norm=1e-3)
assert seen[0] <= 1e-3 * (1 + 1e-4), f"step 之前的梯度范数应不超过 clip_norm，得到 {seen[0]:.2e}"

# (3) BPTT 只 backward 一次：W_h 上的 hook 每个 batch 只触发一次（梯度已经累加了所有时间步）
calls = []
m3 = copy.deepcopy(m)
handle = m3.W_h.register_hook(lambda g: calls.append(1))
loader_2b = DataLoader(TensorDataset(Xs_[:128], ys_[:128]), batch_size=64)
train_epoch(m3, loader_2b, bce, torch.optim.SGD(m3.parameters(), lr=0.0), clip_norm=1.0)
handle.remove()
assert len(calls) == 2, f"2 个 batch 应该只触发 2 次 backward，得到 {len(calls)} 次"

# (4) 能学习：几个 epoch 后 loss 下降、参数被更新
m4 = SimpleRNN(); w0 = m4.W_h.detach().clone()
tr_loader, va_loader, _, _ = make_loaders(10)
opt4 = torch.optim.Adam(m4.parameters(), lr=LEARNING_RATE)
l0, a0, _ = train_epoch(m4, tr_loader, bce, opt4, CLIP_NORM)
l1, a1, _ = train_epoch(m4, tr_loader, bce, opt4, CLIP_NORM)
assert not torch.equal(w0, m4.W_h), "参数没有更新"
assert l1 < l0, f"第二个 epoch 的 loss 应低于第一个：{l0:.4f} → {l1:.4f}"

# (5) evaluate 不改参数、不留梯度；加权平均正确
for p in m4.parameters():
    p.grad = None
snap = copy.deepcopy(m4.state_dict())
e1 = evaluate(m4, va_loader, bce); e2 = evaluate(m4, va_loader, bce)
assert e1 == e2 and all(torch.equal(snap[k], v) for k, v in m4.state_dict().items())
assert all(p.grad is None for p in m4.parameters()), "evaluate 应该在 no_grad 里"
Xv, yv = next(iter(DataLoader(va_loader.dataset, batch_size=len(va_loader.dataset))))
with torch.no_grad():
    lg = m4(Xv)
np.testing.assert_allclose(e1[0], bce(lg, yv).item(), rtol=1e-5)
np.testing.assert_allclose(e1[1], ((lg > 0).float() == yv).float().mean().item(), atol=1e-9)
print(f"Lab 4 自检通过：裁剪前范数、裁剪效果、单次 backward、学习与评估都正确（loss {l0:.4f} → {l1:.4f}）。")

## Lab 5：序列长度实验

下面的 `run_experiment` 已提供（它调用你写的 `SimpleRNN`、`train_epoch` 和 `evaluate`）。每次都从头初始化一个新模型，在固定种子下训练 20 轮，记录每个 epoch 的 loss、验证准确率和裁剪前梯度范数的平均值。

**验收**：

1. $T=10$：验证准确率 > 95%。
2. $T=30$ 和 $T=100$：记录最终准确率，以及你观察到的训练难度（哪个 epoch 开始学会？loss 是否一直停在 0.693 附近？）。**这两个不设通过线**，是实验记录。

> $\ln 2\approx0.693$ 是“随机猜”的 BCE loss。loss 一直停在 0.69 左右、准确率停在 50% 左右，说明模型还没有学到可用的信号（也可能只是处在 plateau，之后才学会，T=30 就可能这样）。

In [ ]:
def run_experiment(T, epochs=EPOCHS, seed=SEED, clip_norm=CLIP_NORM, verbose_every=5):
    """从头训练一个 SimpleRNN，返回历史记录和训练好的模型（已提供）。"""
    train_loader, val_loader, test_loader, _ = make_loaders(T, seed=seed)
    torch.manual_seed(seed)
    model = SimpleRNN()
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    hist = {"train_loss": [], "val_acc": [], "grad_norm": []}
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc, norms = train_epoch(model, train_loader, loss_fn, optimizer, clip_norm)
        va_loss, va_acc = evaluate(model, val_loader, loss_fn)
        hist["train_loss"].append(tr_loss); hist["val_acc"].append(va_acc); hist["grad_norm"].append(float(np.mean(norms)))
        if epoch % verbose_every == 0 or epoch == 1:
            print(f"T={T:>3} epoch {epoch:>2}/{epochs}  train loss {tr_loss:.4f}  val acc {va_acc:.4f}  mean pre-clip grad norm {np.mean(norms):.4f}  ({time.time() - t0:.0f}s)")
    te_loss, te_acc = evaluate(model, test_loader, loss_fn)
    hist["test_acc"] = te_acc
    return hist, model


results, models = {}, {}
for T in T_LIST:
    results[T], models[T] = run_experiment(T)
    print(f"→ T={T}: 验证准确率 {results[T]['val_acc'][-1]:.4f}，测试准确率 {results[T]['test_acc']:.4f}\n")

assert results[10]["val_acc"][-1] > 0.95, "T=10 的验证准确率应超过 95%，回头检查 forward / 训练流程"
print("验收：T=10 通过。")

In [ ]:
# 观察图：三种序列长度的训练过程
epochs_axis = np.arange(1, EPOCHS + 1)
styles = {10: ("-", "o"), 30: ("--", "s"), 100: (":", "^")}
fig, axes = plt.subplots(1, 3, figsize=(15, 4.0))
for T, color in zip(T_LIST, SERIES_COLORS):
    ls, mk = styles[T]
    axes[0].plot(epochs_axis, results[T]["train_loss"], ls, marker=mk, ms=4, color=color, label=f"T={T}")
    axes[1].plot(epochs_axis, results[T]["val_acc"], ls, marker=mk, ms=4, color=color, label=f"T={T}")
    axes[2].plot(epochs_axis, results[T]["grad_norm"], ls, marker=mk, ms=4, color=color, label=f"T={T}")
axes[0].axhline(np.log(2), color="#999", lw=1, ls="-."); axes[0].text(8, np.log(2) + 0.02, "ln 2：随机猜", fontsize=8, color="#666")
axes[0].set_title("训练 loss"); axes[0].set_ylabel("BCE loss")
axes[1].axhline(0.5, color="#999", lw=1, ls="-."); axes[1].axhline(0.95, color="#999", lw=1, ls=":")
axes[1].set_title("验证准确率（0.5 = 随机，0.95 = 验收线）"); axes[1].set_ylim(0.45, 1.02)
axes[2].set_yscale("log"); axes[2].set_title("每个 epoch 的平均裁剪前梯度范数（对数坐标）")
for a in axes:
    a.set_xlabel("epoch"); a.set_xticks(range(2, EPOCHS + 1, 2)); a.legend()
fig.tight_layout(); plt.show()

In [ ]:
# 观察图：训练之后，每个时间步的梯度范数（对数坐标）。对比 T=10 / 30 / 100 的“记忆距离”。
fig, ax = plt.subplots(figsize=(8, 4.2))
for T, color in zip(T_LIST, SERIES_COLORS):
    Xg, yg = generate_sequences(256, T, seed=99)
    g_norms = hidden_grad_norms(models[T], Xg, yg, nn.BCEWithLogitsLoss())
    ax.plot(np.arange(1, T + 1) - T, g_norms.numpy() + 1e-30, styles[T][0], color=color, lw=2, label=f"T={T}（训练后）")
ax.set_yscale("log"); ax.set_xlabel("距离最后一步的时间步数（0 = 最后一步，-(T-1) = 第一个数字）")
ax.set_ylabel(r"$\|\partial L/\partial h_t\|$（对数坐标）")
ax.set_title("训练后各时间步的梯度范数"); ax.legend(); fig.tight_layout(); plt.show()

print(f"{'T':>4} {'最终 train loss':>16} {'验证准确率':>12} {'测试准确率':>12} {'首次验证准确率 > 0.9 的 epoch':>28}")
for T in T_LIST:
    first = next((i + 1 for i, a in enumerate(results[T]["val_acc"]) if a > 0.9), None)
    print(f"{T:>4} {results[T]['train_loss'][-1]:>16.4f} {results[T]['val_acc'][-1]:>12.4f} {results[T]['test_acc']:>12.4f} {str(first):>28}")

### 记录你的观察

运行上面的实验后，把数字填进表里，再回答下面的问题。**同一个 seed 的结果会随着随机数库版本略有不同**，以你自己运行的结果为准。

| T | 最终验证准确率 | 测试准确率 | 首次验证准确率 > 0.9 的 epoch | 训练难度（容易 / 有 plateau / 学不会） |
| --- | --- | --- | --- | --- |
| 10 | | | | |
| 30 | | | | |
| 100 | | | | |

**问题（写在这里）**

1. T=30 的 loss 曲线是怎样的？它在学会之前有一段停在 0.693 附近的 plateau 吗？这段时间模型在做什么？
2. T=100 在 20 轮里学会了吗？如果没有，把 `EPOCHS` 改成 100 再跑一次 `run_experiment(100, epochs=100)`，会怎样？
3. 对比第 3 张图里 T=10 / 30 / 100 训练后的梯度范数曲线，哪个模型的梯度能传回第一个时间步？
4. 平均裁剪前梯度范数在训练过程中的变化趋势是什么？它有没有超过 `CLIP_NORM`？裁剪在这个实验里起了多大作用？

你的回答：

## 进阶研究清单

> **TODO（研究）**
>
> - [ ] **梯度裁剪消融**：把 `CLIP_NORM` 设成 `1e9`（等于不裁剪），对比 T=30 的训练稳定性。设成 0.1 又如何？
> - [ ] **初始化**：把 `W_h` 初始化成**正交矩阵**（`nn.init.orthogonal_`）或单位矩阵，T=100 能学会吗？为什么？提示：看 4.5 里 $\|W_h\|$ 的作用。
> - [ ] **用 ReLU 代替 tanh**：对比 T=30 的表现，了解 IRNN（Le et al., 2015）。
> - [ ] **Truncated BPTT**：只让梯度回传最近 $k$ 步（在循环里每隔 $k$ 步 `h = h.detach()`）。第一个数字离终点 $T-1$ 步，截断后还学得会吗？
> - [ ] **计算图的内存**：T=100 和 T=1000 的显存 / 内存占用有什么差别？为什么 BPTT 的内存随 $T$ 线性增长？
> - [ ] **阅读**：[Pascanu et al., *On the difficulty of training recurrent neural networks*, 2013](https://arxiv.org/abs/1211.5063)（梯度消失 / 爆炸与梯度裁剪）、[Understanding LSTM Networks (Olah, 2015)](https://colah.github.io/posts/2015-08-Understanding-LSTMs/)。
> - [ ] **下一步**：[04_lstm_memory_pytorch](../04_lstm_memory_pytorch/README.md) 在同一个任务上把 RNN cell 换成 LSTM cell，你可以把这里的 `results` 数字带过去对比。